# 03 — Inference & Visualization

This notebook:
1. Loads the trained delta model
2. Evaluates on all 5 benchmarks (baseline vs delta)
3. Generates visual comparison figures
4. Produces the final results summary

In [ ]:
import sys, os
sys.path.insert(0, os.path.dirname(os.getcwd()))

import numpy as np
import torch
from src.models.cfsr import load_cfsr_model
from src.models.cfsr_delta import load_delta_model
from src.metrics.sr_metrics import evaluate_all_benchmarks, evaluate_dataset, calc_psnr
from src.utils.visualization import (
    load_image, to_tensor, to_numpy, create_comparison, save_comparison_figure
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

## Configuration

In [ ]:
BACKBONE_WEIGHTS = '../model_zoo/CFSR_x4.pth'
REFINE_WEIGHTS = '../checkpoints/refine_best_x4.pth'  # or delta_refine_v3/best.pth
BENCH_DIR = '../datasets/benchmark'
OUTPUT_DIR = '../results/figures'
os.makedirs(OUTPUT_DIR, exist_ok=True)
SCALE = 4

## Load Models

In [ ]:
# Baseline
baseline_model, base_params = load_cfsr_model(SCALE, BACKBONE_WEIGHTS, str(device))
print(f'Baseline: {base_params:,} params')

# Delta
delta_model, delta_counts = load_delta_model(
    SCALE, BACKBONE_WEIGHTS, REFINE_WEIGHTS, device=str(device)
)
print(f'Delta: {delta_counts["total"]:,} params ({delta_counts["trainable"]:,} trainable)')

## Full Benchmark Evaluation

In [ ]:
base_results = evaluate_all_benchmarks(baseline_model, SCALE, BENCH_DIR, str(device))
delta_results = evaluate_all_benchmarks(delta_model, SCALE, BENCH_DIR, str(device))

print(f"{'Dataset':<12} {'Base PSNR':<12} {'Delta PSNR':<12} {'Gain':<10} {'Base SSIM':<12} {'Delta SSIM':<12}")
print('-' * 70)
for ds in ['Set5', 'Set14', 'B100', 'Urban100', 'Manga109']:
    if ds in base_results and ds in delta_results:
        bp = base_results[ds]['psnr']; dp = delta_results[ds]['psnr']
        bs = base_results[ds]['ssim']; ds_ = delta_results[ds]['ssim']
        g = dp - bp
        print(f'{ds:<12} {bp:<12.4f} {dp:<12.4f} {g:<+10.4f} {bs:<12.4f} {ds_:<12.4f}')

## Visual Comparison — Set5

Change `IMAGE_NAME` to visualize any Set5 image:
`baby`, `bird`, `butterfly`, `head`, `woman`

In [ ]:
IMAGE_NAME = 'butterfly'  # <-- change this

lr_path = os.path.join(BENCH_DIR, 'Set5', f'LR_bicubic/X{SCALE}', f'{IMAGE_NAME}x{SCALE}.png')
hr_path = os.path.join(BENCH_DIR, 'Set5', 'HR', f'{IMAGE_NAME}.png')

lr_img = load_image(lr_path)
hr_img = load_image(hr_path)
lr_tensor = to_tensor(lr_img).to(device)

with torch.no_grad():
    base_sr = to_numpy(baseline_model(lr_tensor))
    delta_sr = to_numpy(delta_model(lr_tensor))

h, w = min(base_sr.shape[0], hr_img.shape[0]), min(base_sr.shape[1], hr_img.shape[1])
bp = calc_psnr(base_sr[:h,:w], hr_img[:h,:w], SCALE)
dp = calc_psnr(delta_sr[:h,:w], hr_img[:h,:w], SCALE)

fig = create_comparison(lr_img, base_sr, delta_sr, hr_img, SCALE, bp, dp)
save_comparison_figure(fig, os.path.join(OUTPUT_DIR, f'comparison_{IMAGE_NAME}.png'))

import matplotlib.pyplot as plt
fig2 = create_comparison(lr_img, base_sr, delta_sr, hr_img, SCALE, bp, dp)
plt.show()
print(f'\n{IMAGE_NAME}: Base={bp:.4f}  Delta={dp:.4f}  Gain={dp-bp:+.4f} dB')

## Generate All Set5 Comparisons

In [ ]:
print('Generating all Set5 comparisons...\n')
for name in ['baby', 'bird', 'butterfly', 'head', 'woman']:
    lr_p = os.path.join(BENCH_DIR, 'Set5', f'LR_bicubic/X{SCALE}', f'{name}x{SCALE}.png')
    hr_p = os.path.join(BENCH_DIR, 'Set5', 'HR', f'{name}.png')
    if not os.path.exists(lr_p): continue
    lr = load_image(lr_p); hr = load_image(hr_p)
    t = to_tensor(lr).to(device)
    with torch.no_grad():
        b_sr = to_numpy(baseline_model(t))
        d_sr = to_numpy(delta_model(t))
    h_, w_ = min(b_sr.shape[0], hr.shape[0]), min(b_sr.shape[1], hr.shape[1])
    b_p = calc_psnr(b_sr[:h_,:w_], hr[:h_,:w_], SCALE)
    d_p = calc_psnr(d_sr[:h_,:w_], hr[:h_,:w_], SCALE)
    fig = create_comparison(lr, b_sr, d_sr, hr, SCALE, b_p, d_p)
    save_comparison_figure(fig, os.path.join(OUTPUT_DIR, f'comparison_{name}.png'))
    print(f'  {name}: Base={b_p:.4f}  Delta={d_p:.4f}  Gain={d_p-b_p:+.4f}')
print(f'\nAll figures saved to {OUTPUT_DIR}')